# R parity matrix

The tables compare the operations available in the Python package with those
in the R binding, and record which of them have been compared numerically
with upstream R `mboost` (the R package mboostrs follows) and `gamboostLSS`
(its extension to distributional models). GLM and GAM mean the linear
(`glmboost`) and additive (`gamboost`) boosted models; LSS (location, scale
and shape) families are distributional families in which each distribution
parameter has its own additive predictor; CV is cross-validation.

The generated tables use the status values of the capability manifest, the
record returned by `mb.family_capabilities()` and `mb.learner_capabilities()`:

- `verified`: compared with a reference implementation or reference value at
  a recorded tolerance.
- `smoke`: tests check that the operation runs and satisfies stated
  invariants, without an external numerical reference.
- `intentional-deviation`: the tested behaviour differs deliberately from the
  upstream package; the validation record states why.
- `unverified`: the manifest makes no correctness claim.

## Core models
```{list-table}
:header-rows: 1
* - Feature
  - Python
  - R binding
  - Matches R `mboost`
* - `glmboost`, `gamboost`
  - Yes
  - Yes
  - Yes (≤1e-6)
* - `cvrisk` (bootstrap/kfold/subsampling)
  - Yes
  - Yes; model weights are combined with fold weights
  - Yes
* - stability selection
  - Yes
  - Yes
  - Yes (error bounds as computed by the R package `stabs`)
* - AIC / logLik / summary / varimp
  - Yes
  - Yes
  - Yes where defined
* - `hatvalues`, `confint`, `plot_risk`, `plot_varimp`
  - Yes
  - Yes
  - Yes
* - `stabsel`
  - Any fitted model
  - Linear only: the subsample fits run through `glmboost`, so a formula with base-learner terms such as `bbs(x)` stops with an error
  - Yes (error bounds as computed by `stabs`)
```
## Base learners
`bols`, `bbs`, `bmono`, `btree`, `brandom`, `bspatial`, `bmrf`, `bns` and
`bkernel` are available in R formulas. `bmrf` takes its neighbourhood matrix
as `K=` (or `bnd=`, the upstream spelling) and looks the named object up in
the formula's environment. `bkernel` in mboostrs is a one-dimensional kernel
ridge learner; upstream `mboost`'s `bkernel` is a kernel over sets of genetic
markers for pathway analysis. A formula written for one fits a different
model under the other, so `bkernel` is recorded as an intentional deviation.

## Families

Tables generated from `mboostrs.family_capabilities()`. The `known_deviation`
column names a documented difference from the R function in `r_name`; an
empty cell means none is recorded.

In [1]:
import mboostrs as mb
import polars as pl

caps = mb.family_capabilities(output="polars")
standard = caps.filter(pl.col("kind").is_in(["standard", "standard_survival", "standard_multiclass"]))
standard.select(["name", "kind", "r_package", "r_name", "known_deviation"])

name,kind,r_package,r_name,known_deviation
str,str,str,str,str
"""AUC""","""standard""","""mboost""","""AUC""",""""""
"""AdaExp""","""standard""","""mboost""","""AdaExp""",""""""
"""Binomial""","""standard""","""mboost""","""Binomial""",""""""
"""Cindex""","""standard_survival""","""mboost""","""Cindex""",""""""
"""CoxPH""","""standard_survival""","""mboost""","""CoxPH""",""""""
…,…,…,…,…
"""Hurdle""","""standard""","""mboost""","""Hurdle""",""""""
"""Poisson""","""standard""","""mboost""","""Poisson""",""""""
"""PropOdds""","""standard""","""mboost""","""PropOdds""",""""""


## Distributional families

[Implementation differences](../implementation/parity_and_bugs) explains the
mechanism of each `known_deviation` entry and names the tests that measure it.

In [2]:
lss = caps.filter(pl.col("kind") == "lss")
lss.select(["name", "r_package", "r_name", "known_deviation"])

name,r_package,r_name,known_deviation
str,str,str,str
"""BCCGLSS""","""gamboostLSS""","""BCCGLSS""",""""""
"""BCPELSS""","""gamboostLSS""","""BCPELSS""",""""""
"""BCTLSS""","""gamboostLSS""","""BCTLSS""",""""""
"""BetaLSS""","""gamboostLSS""","""BetaLSS""",""""""
"""BinomialLSS""","""gamboostLSS""","""BinomialLSS""",""""""
…,…,…,…
"""WeibullLSS""","""gamboostLSS""","""WeibullLSS""","""Auto offset deliberately fixes…"
"""ZINBLSS""","""gamboostLSS""","""ZINBLSS""",""""""
"""ZIPoLSS""","""gamboostLSS""","""ZIPoLSS""",""""""


Rows with `kind == "standard_missing"` identify upstream families or
helpers that mboostrs has not implemented:


In [3]:
caps.filter(pl.col("kind") == "standard_missing").select(["name", "r_package", "notes"])

name,r_package,notes
str,str,str
"""FP""","""mboost""","""Known upstream mboost family/h…"
"""Family""","""mboost""","""Known upstream mboost family/h…"
"""IPCweights""","""mboost""","""Not a boosting family (no fit/…"
"""RCG""","""mboost""","""Known upstream mboost family/h…"


### Distributional fitting mechanics

```{list-table}
:header-rows: 1
* - Feature
  - Python
  - R binding
  - Note
* - cyclic fitting
  - Yes
  - Yes
  - `Bols`, difference-penalized `Bbs`, ridge-penalized `Bns`, random-intercept `Brandom`, graph-smoothed `Bmrf`, tensor-product `Bspatial`, shape-constrained `Bmono`, caller-constrained `Bconstrained`, and tree-based `Btree` may be mixed within each parameter
* - non-cyclic fitting
  - mixed `Bols`/`Bbs`/`Bns`/`Brandom`/`Bmrf`/`Bspatial`/`Bmono`/`Bconstrained`/`Btree`
  - mixed `Bols`/`Bbs`/`Bns`/`Brandom`/`Bmrf`/`Bspatial`/`Bmono`/`Bconstrained`/`Btree`
  - `gamboostLSS(method="noncyclic")` selects one parameter-qualified learner per global step
* - `cvrisk` (per-param mstop grid)
  - Yes
  - Yes
  - mixed structured learners; smoothing parameters and factorizations are recomputed from each fold's in-bag weights
* - stability selection
  - Yes
  - Yes
  - `stabsel_glmboostLSS` is linear; `stabsel_gamboostLSS` tracks every supported mixed learner with parameter-qualified identities
```
Rows that name a known deviation are implemented independently in `mboostrs` and differ from R; see [](../implementation/parity_and_bugs).


## Multinomial
```{list-table}
:header-rows: 1
* - Feature
  - Python
  - R binding
* - `bols` (linear) multinomial
  - Yes
  - Yes (matches R `mboost` ~1e-13)
* - `bbs` (smooth) multinomial
  - Yes
  - Yes (matches R `mboost` ~1e-6)
* - predict response matrix, in-sample
  - Yes
  - Yes
* - predict newdata / truncated mstop
  - Yes
  - Yes
```
Both frontends fit multinomial `gamboost` with numeric `bols`, `bbs` and a
`brandom` over a factor. For each such term the binding builds the learner
that upstream `mboost` writes as `bols(x) %O% bols(class)` (or `bbs(x) %O%
bols(class)`): the Kronecker product of the term with a class indicator, so
each class gets its own coefficients. The user writes `y ~ bols(x)` with
`family = Multinomial()`; typing `%O%` in an R formula stops with an error,
as any infix learner operator does in the R frontend. Prediction on new data
and at a truncated `mstop` is available in both frontends.

## Multinomial limitations

- A `bols` term over a factor covariate, and `btree` and `bmono` terms, stop
  with "Multinomial currently supports bols(), bbs() and brandom()
  base-learners" in both frontends. For a factor covariate, use
  `brandom(factor)`.

## Mixed distributional learners

A distributional model may mix `Bols`, `Bbs`, `Bns`, `Brandom`, `Bmrf`,
`Bspatial`, `Bmono`, `Bconstrained`, `Btree`, `Bkernel` and `Buser` terms in
each parameter. In the list below, *replay* means rebuilding a learner's
design for new rows from what was stored at fit time.

- `Brandom`, `Bmrf`, `Bspatial`, `Bmono` and `Btree`: fits and link-scale
  predictions are compared with R `gamboostLSS` at a tolerance of 1e-10.
- `Buser`: fits and link-scale predictions are compared with `gamboostLSS`;
  tests in both frontends check replay of the user-supplied matrix and its
  use under resampling.
- `Bconstrained`: an mboostrs learner whose constraint matrix the caller
  supplies; tests check that the fitted coefficients satisfy the constraints
  and that replay reproduces the fit.
- `Bkernel`: the mboostrs kernel ridge learner; tests check that it fits,
  replays and resamples. It is not compared with upstream, whose `bkernel` is
  a different construction.
- Every learner: tests in both frontends check replay, CV, stability
  selection and saving and loading. For the grouped learners (`brandom`,
  `bmrf`), they also check prediction for a group level that was absent from
  the training data.

`gamboostLSS` accepts numeric and factor `by=` terms (varying coefficients:
the term's effect is multiplied by, or split by, the `by` variable) on `bbs`
and `bols`. In Python, distributional models accept a numeric `by=` on
every learner and a factor `by=` on `bbs` and `bols`, and predict new data
with those terms.
## Additional capabilities
```{list-table}
:header-rows: 1
* - Feature
  - Python
  - R binding
  - Note
* - Prediction contributions (`which=`, `aggregate=`)
  - Yes
  - —
  - Per-learner decomposition; GLM + GAM
* - Extraction methods (`variable_names`, `model_frame`, etc.)
  - Yes
  - —
  - R equivalent: `extract(what=...)`, `variable.names`, `model.frame`
* - Structured `summary(as_dict=True)`
  - Yes
  - —
  - JSON-serializable
* - Weighted LSS fits (`glmboostLSS(weights=)`)
  - Yes
  - —
  - Observation weights for distributional models
* - LSS GAIC (generalized Akaike information criterion) and logLik
  - Yes
  - Yes
  - The degrees of freedom are the number of boosting updates across all parameters plus one; `gaic(k=2)` uses the same count as R `logLik.mboostrs_glmboostLSS`
* - Model comparison (`gaic_table`, `compare_models`)
  - Yes
  - —
  - Cross-model GAIC/risk comparison
* - Family scanning (`scan_lss_families`)
  - Yes
  - —
  - Collects failures instead of crashing
* - mstop profiling (`profile_mstop`)
  - Yes
  - —
  - Risk/GAIC at mstop grid points
* - Comparison with R (`compare_to_r`)
  - Yes
  - —
  - Fits the same specification in mboostrs and R `mboost` and reports the differences
* - Distribution functions (pdf/cdf/quantile/simulate)
  - Yes
  - Yes
  - Of the 27 LSS families: density or mass for 26, CDF and quantile for 25, simulation for 14 (columns `pdf_or_pmf`, `cdf`, `quantile` and `simulate` of `family_capabilities()`)
* - Diagnostics module (`quantile_residuals`, `pit`, `q_stats`, etc.)
  - Yes
  - —
  - Works for `glmboost`, `gamboost` and LSS models
```

## Numerical comparisons

The capability manifest names the tests behind each family's status in each
frontend. The table lists those records, using the status values defined at
the top of the page; `validation_ids` names the tests.

In [4]:
fit_evidence = caps.filter(pl.col("fit") == "yes").select(
    ["name", "kind", "python_fit_status", "r_fit_status", "validation_ids"]
).sort(["r_fit_status", "kind", "name"])
fit_evidence

name,kind,python_fit_status,r_fit_status,validation_ids
str,str,str,str,str
"""LogLogLSS""","""lss""","""intentional-deviation""","""intentional-deviation""","""deviation.loglog_offset; py.fa…"
"""WeibullLSS""","""lss""","""intentional-deviation""","""intentional-deviation""","""deviation.weibull_offset; py.f…"
"""Gehan""","""standard_survival""","""intentional-deviation""","""intentional-deviation""","""deviation.gehan_offset; parity…"
"""GaussianKnownVarLSS""","""lss""","""smoke""","""smoke""","""parity.r.gaussianknownvarlss; …"
"""ZINBLSS""","""lss""","""unverified""","""smoke""","""parity.r.zinblss; py.family.co…"
…,…,…,…,…
"""StudentTLSS""","""lss""","""verified""","""verified""","""parity.python.studenttlss; par…"
"""ZIPoLSS""","""lss""","""verified""","""verified""","""parity.python.zipolss; parity.…"
"""Gaussian""","""standard""","""verified""","""verified""","""parity.python.gaussian; parity…"


The [R parity evidence workflow](https://github.com/fcela/mboostrs/actions/workflows/parity.yml) runs all tests carrying the `r_parity` marker every Monday and on manual request. Pull requests that change families, learners, cross-validation, distributional models, prediction or the bindings run a fixed subset of those tests. Each run uploads `parity-results.json`, conforming to [`parity-result.schema.json`](../../schemas/parity-result.schema.json), with node status, measured absolute and relative error, the tolerance used by the numeric comparison, package versions, platform, and commit.

A test that does not report its numeric comparison has
`measurement_status = "not_emitted"` and null error and tolerance values; a
null means the measurement was not recorded.